# Ramsey County Racial Covenants Table

Assembled by Alexander Price with the help of Claude Sonnet 4.6

This code was made using Claude Sonnet 4.6 AI because there was no way of getting a table like the Hennepin County table because it was published and cleaned ready for use. For Ramsey County it was still being processed and the only files were geo files that were meant for ArcGIS, so I did go into ArcGIS and get the coordinate geometry and projected coordinate system for each polygon for each house in the shapefile attributes table and used to put on here.

### Uploading the shapefile folder for Ramsey County

All five files (`.shp`, `.dbf`, `.prj`, `.shx`, `.cpg`) have to be in the same folder together with the shapefile for it used the other files for its attribute table, projected coordinate system, database file, and encoding to use in geo pandas and jupyter notebook. 

In [1]:
import geopandas as gpd
import pandas as pd
import re
from datetime import datetime

gdf = gpd.read_file("covenants-mn-ramsey-county.shp")

print(f"Rows loaded : {len(gdf):,}")
print(f"CRS         : {gdf.crs}")
print(f"Geometry    : {gdf.geometry.geom_type.unique()}")
print(f"Columns     : {list(gdf.columns)}")

Rows loaded : 5,528
CRS         : EPSG:4326
Geometry    : ['Polygon' 'MultiPolygon']
Columns     : ['db_id', 'workflow', 'cnty_name', 'cnty_fips', 'doc_num', 'deed_year', 'deed_date', 'exec_date', 'cov_text', 'seller', 'buyer', 'street_add', 'city', 'state', 'zip_code', 'add_cov', 'block_cov', 'lot_cov', 'map_book', 'map_page', 'cnty_pin', 'add_mod', 'block_mod', 'lot_mod', 'ph_dsc_mod', 'join_strgs', 'geocd_addr', 'geocd_dist', 'cov_type', 'match_type', 'manual_cx', 'dt_updated', 'zn_subj_id', 'zn_dt_ret', 'image_ids', 'med_score', 'plat_dbid', 'subd_dbid', 'geometry']


### Extract centroid X and Y coordinates from the shapefile 
- To use the shapefile have to use geopandas to project the coordinate system (WGS84) to match up with the addresses with calculating the center point of each of the address polygons in the shapefile. The WGS84 is the same coordinate projection that Hennepin County table has. 

In [2]:
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    print("Reprojected to WGS84")
else:
    print("Already in WGS84 — no reprojection needed")

gdf["X"] = gdf.geometry.centroid.x.round(8)
gdf["Y"] = gdf.geometry.centroid.y.round(8)

print(f"X range: {gdf['X'].min()} to {gdf['X'].max()}")
print(f"Y range: {gdf['Y'].min()} to {gdf['Y'].max()}")
gdf[["street_add", "X", "Y"]].head()

Already in WGS84 — no reprojection needed
X range: -93.2276125 to -92.9914938
Y range: 44.89868046 to 45.12424092


C:\Users\price\AppData\Local\Temp\ipykernel_46588\1964633884.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["X"] = gdf.geometry.centroid.x.round(8)
C:\Users\price\AppData\Local\Temp\ipykernel_46588\1964633884.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["Y"] = gdf.geometry.centroid.y.round(8)


,street_add,X,Y
0,1896 KENNARD ST N,-93.029640,44.998420
1,1826 FLANDRAU ST N,-93.027122,44.996089
2,1838 FLANDRAU ST N,-93.027122,44.996500
3,1919 FLANDRAU ST N,-93.028351,44.998701
4,1899 WHITE BEAR AVE N,-93.025984,44.998140


### Helper functions for date formatting, workflow mapping, and address formatting

These functions convert Ramsey County field values to match the Hennepin County format.

In [ ]:
def fmt_deed_date(val):
    if pd.isna(val) or str(val).strip() == "":
        return ""
    s = str(val).split(" ")[0].strip()
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%m/%d/%y"):
        try:
            dt = datetime.strptime(s, fmt)
            return f"{dt.month}/{dt.day}/{dt.year}"
        except ValueError:
            pass
    return s


def map_workflow(cov_type):
    ct = str(cov_type).lower() if pd.notna(cov_type) else ""
    if "zooniverse" in ct:
        return "Zooniverse -- Abstracts"
    if "manual" in ct:
        return "Manual"
    return str(cov_type) if pd.notna(cov_type) else ""


def title_fix(s):
    if pd.isna(s) or str(s).strip() == "":
        return ""
    s = str(s).strip().title()
    s = re.sub(r"\b(N|S|E|W|Ne|Nw|Se|Sw)\b", lambda m: m.group().upper(), s)
    return s


def fmt_address(street, city, zip_code):
    parts = [
        title_fix(street),
        title_fix(city),
        str(zip_code).strip() if pd.notna(zip_code) else ""
    ]
    return ", ".join(p for p in parts if p)


def coalesce(*args):
    for a in args:
        if pd.notna(a) and str(a).strip():
            return str(a).strip()
    return ""



Helper functions defined.


### We build the Hennepin-formatted table from the shapefile attributes

Each column is mapped from the Ramsey source fields to match the Hennepin County schema. The `_mod` columns are the cleaned versions and are used first, falling back to the raw `_cov` columns where needed.

In [4]:
df = pd.DataFrame()

df["FID"]       = gdf["db_id"].fillna("").astype(str).str.replace(r"\.0+$", "", regex=True)
df["Doc_ID"]    = gdf["doc_num"].fillna("").astype(str).str.strip()
df["Deed_ID"]   = gdf.apply(
    lambda r: f"{str(r.get('map_book','')).strip()}_{str(r.get('map_page','')).strip()}"
    if coalesce(r.get("map_book")) and coalesce(r.get("map_page")) else "", axis=1)

df["Racial_Res"] = gdf["cov_text"].fillna("").str.strip()
df["Type_Res"]   = "NA"

df["Addition"]  = gdf.apply(lambda r: coalesce(r.get("add_mod"),   r.get("add_cov")),   axis=1)
df["City"]      = gdf["city"].fillna("").str.strip()
df["Block"]     = gdf.apply(lambda r: coalesce(r.get("block_mod"), r.get("block_cov")), axis=1)
df["Lot"]       = gdf.apply(lambda r: coalesce(r.get("lot_mod"),   r.get("lot_cov")),   axis=1)

df["Grantor"]   = gdf["seller"].fillna("").str.strip()
df["Grantee"]   = gdf["buyer"].fillna("").str.strip()

df["Date_Deed"] = gdf["deed_date"].apply(fmt_deed_date)
df["Date_Ex"]   = ""
df["Ex_Year"]   = ""

df["Join_ID"]   = gdf.apply(
    lambda r: " ".join(filter(None, [
        str(r.get("add_mod",   "") or "").strip(),
        str(r.get("block_mod", "") or "").strip(),
        str(r.get("lot_mod",   "") or "").strip()
    ])), axis=1)

df["Workflow"]  = gdf["cov_type"].apply(map_workflow)
df["Rel_Score"] = gdf["med_score"].fillna("").astype(str).str.strip()

df["X"]         = gdf["X"]
df["Y"]         = gdf["Y"]

df["Address"]   = gdf.apply(
    lambda r: fmt_address(r.get("street_add"), r.get("city"), r.get("zip_code")), axis=1)

df["Distance"]  = ""

print(f"Rows built : {len(df):,}")
df.head(3)

Rows built : 5,528


,FID,Doc_ID,Deed_ID,Racial_Res,Type_Res,Addition,City,Block,Lot,Grantor,...,Date_Deed,Date_Ex,Ex_Year,Join_ID,Workflow,Rel_Score,X,Y,Address,Distance
0,2882191,897801,,"That said premises and lots, or any of them, s...",NA,GARDEN ACRES,MAPLEWOOD,2,48,Northline Corporation,...,5/11/1936,,,GARDEN ACRES 2 48,Manual,,-93.029640,44.998420,"1896 Kennard St N, Maplewood, 55109",
1,3034324,897801,,"That said premises and lots, or any of them, s...",NA,GARDEN ACRES,MAPLEWOOD,1,40,Trustees of Morris Fink Estate,...,5/11/1936,,,GARDEN ACRES 1 40,Manual,,-93.027122,44.996089,"1826 Flandrau St N, Maplewood, 55109",
2,3034320,897801,,"That said premises and lots, or any of them, s...",NA,GARDEN ACRES,MAPLEWOOD,1,41,Northline Corporation,...,5/11/1936,,,GARDEN ACRES 1 41,Manual,,-93.027122,44.996500,"1838 Flandrau St N, Maplewood, 55109",


### Validate the output

In [5]:
print("=== Validation Summary ===")
print(f"Total rows          : {len(df):,}")
print(f"Rows with X/Y       : {df['X'].notna().sum():,}")
print(f"Rows with Address   : {(df['Address'] != '').sum():,}")
print(f"Rows with Racial_Res: {(df['Racial_Res'] != '').sum():,}")
print(f"Rows with Date_Deed : {(df['Date_Deed'] != '').sum():,}")
print(f"\nWorkflow breakdown:")
print(df["Workflow"].value_counts().to_string())
print(f"\nX range : {df['X'].min()} → {df['X'].max()}")
print(f"Y range : {df['Y'].min()} → {df['Y'].max()}")
print(f"\nSample addresses:")
for a in df["Address"].head(5).tolist():
    print(f"  {a}")

=== Validation Summary ===
Total rows          : 5,528
Rows with X/Y       : 5,528
Rows with Address   : 5,528
Rows with Racial_Res: 5,528
Rows with Date_Deed : 5,528

Workflow breakdown:
Workflow
Manual                     3164
Zooniverse -- Abstracts    2364

X range : -93.2276125 → -92.9914938
Y range : 44.89868046 → 45.12424092

Sample addresses:
  1896 Kennard St N, Maplewood, 55109
  1826 Flandrau St N, Maplewood, 55109
  1838 Flandrau St N, Maplewood, 55109
  1919 Flandrau St N, Maplewood, 55109
  1899 White Bear Ave N, Maplewood, 55109


In [7]:
df_ramsey = df.copy()

In [8]:
display(df_ramsey.info())
display(df_ramsey.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5528 entries, 0 to 5527
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   FID         5528 non-null   object 
 1   Doc_ID      5528 non-null   object 
 2   Deed_ID     5528 non-null   object 
 3   Racial_Res  5528 non-null   object 
 4   Type_Res    5528 non-null   object 
 5   Addition    5528 non-null   object 
 6   City        5528 non-null   object 
 7   Block       5528 non-null   object 
 8   Lot         5528 non-null   object 
 9   Grantor     5528 non-null   object 
 10  Grantee     5528 non-null   object 
 11  Date_Deed   5528 non-null   object 
 12  Date_Ex     5528 non-null   object 
 13  Ex_Year     5528 non-null   object 
 14  Join_ID     5528 non-null   object 
 15  Workflow    5528 non-null   object 
 16  Rel_Score   5528 non-null   object 
 17  X           5528 non-null   float64
 18  Y           5528 non-null   float64
 19  Address     5528 non-null  

None

,FID,Doc_ID,Deed_ID,Racial_Res,Type_Res,Addition,City,Block,Lot,Grantor,...,Date_Deed,Date_Ex,Ex_Year,Join_ID,Workflow,Rel_Score,X,Y,Address,Distance
0,2882191,897801,,"That said premises and lots, or any of them, s...",NA,GARDEN ACRES,MAPLEWOOD,2,48,Northline Corporation,...,5/11/1936,,,GARDEN ACRES 2 48,Manual,,-93.029640,44.998420,"1896 Kennard St N, Maplewood, 55109",
1,3034324,897801,,"That said premises and lots, or any of them, s...",NA,GARDEN ACRES,MAPLEWOOD,1,40,Trustees of Morris Fink Estate,...,5/11/1936,,,GARDEN ACRES 1 40,Manual,,-93.027122,44.996089,"1826 Flandrau St N, Maplewood, 55109",
2,3034320,897801,,"That said premises and lots, or any of them, s...",NA,GARDEN ACRES,MAPLEWOOD,1,41,Northline Corporation,...,5/11/1936,,,GARDEN ACRES 1 41,Manual,,-93.027122,44.996500,"1838 Flandrau St N, Maplewood, 55109",
3,2914680,897801,,"That said premises and lots, or any of them, s...",NA,GARDEN ACRES,MAPLEWOOD,2,2,Trustees of Morris Fink Estate,...,5/11/1936,,,GARDEN ACRES 2 2,Manual,,-93.028351,44.998701,"1919 Flandrau St N, Maplewood, 55109",
4,3035078,897801,,"That said premises and lots, or any of them, s...",NA,GARDEN ACRES,MAPLEWOOD,1,4,Trustees of Morris Fink Estate,...,5/11/1936,,,GARDEN ACRES 1 4,Manual,,-93.025984,44.998140,"1899 White Bear Ave N, Maplewood, 55109",
